# Milestone 2 — target construction and decision-oriented EDA

This notebook is deliberately thin. Every number and figure below is computed by
the project package, not by code written here, so nothing in the repository can
depend on a notebook having been run in the right order:

- the target is built by `california_housing_pulse.features.target`, from the
  frozen contract in `configs/target.yaml`;
- the measurements come from `california_housing_pulse.eda.analysis`;
- the figures come from `california_housing_pulse.eda.figures`.

The generated report is `reports/eda.md`; the judgement drawn from it is in
`docs/MILESTONE_2_EDA_MEMO.md`. Rebuild everything with `make all`.

In [ ]:
import pandas as pd

from california_housing_pulse.eda import analysis
from california_housing_pulse.features.target import load_contract, modeling_rows
from california_housing_pulse.io import read_parquet
from california_housing_pulse.paths import PROCESSED_DIR

pd.set_option("display.width", 160)

panel = read_parquet(PROCESSED_DIR / "county_month_panel.parquet")
model = modeling_rows(panel)
contract = load_contract()

print(contract.describe())
print(f"{len(model):,} modelling rows of {len(panel):,} panel rows")

## 1. Is the target learnable?

A centred, roughly symmetric distribution with no class starved of examples.

In [ ]:
distribution = analysis.describe_target(model)
print(f"mean {distribution.mean:+.3f} pp   sd {distribution.std:.3f} pp")
print(f"IQR {distribution.iqr:.2f} pp   median |Δg| {distribution.abs_median:.2f} pp")

analysis.prevalence_overall(model, contract)

![Target distribution](../reports/figures/fig01_target_distribution.png)

## 2. Does the class mix hold across time?

It does not — and that is the finding. 2022 and 2023 are near mirror images.

In [ ]:
analysis.prevalence_by_year(model, contract)

![Class prevalence by year](../reports/figures/fig02_class_prevalence_by_year.png)

## 3. Why four counties are excluded

Target volatility tracks market thinness, not market drama.

In [ ]:
print("Excluded by the volume floor:")
display(analysis.excluded_county_dispersion(panel))

print("Prevalence by volume tier — note how 'stable' collapses in thin counties:")
analysis.prevalence_by_tier(model, contract)

![Volume versus dispersion](../reports/figures/fig04_volume_vs_dispersion.png)

![County time series](../reports/figures/fig03_county_time_series.png)

## 4. Seasonality and structural breaks

No calendar-month effect worth a feature; large and abrupt regime shifts.

In [ ]:
display(analysis.seasonality(model))
display(analysis.regime_shifts(model))
analysis.rate_regime_prevalence(model, contract)

![Seasonality](../reports/figures/fig05_seasonality.png)

---

Findings, the decisions they drive, and the remaining risks are written up in
[`docs/MILESTONE_2_EDA_MEMO.md`](../docs/MILESTONE_2_EDA_MEMO.md).